# Day 5 — Neural Networks & ANN (Keras)
## RetailMind AI: Customer Churn Propensity Model

**Goal:** Build a feed-forward Artificial Neural Network (ANN) for Telco Customer Churn, compare it fairly with your Day 3/4 classical ML champion, and save the trained model.

### What you will learn
1. TensorFlow/Keras basics using the MNIST handwritten-digit demo.
2. Tabular-data preprocessing for a neural network.
3. Building a Dense ANN with ReLU and sigmoid.
4. Training with EarlyStopping and monitoring validation loss.
5. Evaluating and comparing ANN with your classical ML champion.
6. Saving the model and recording an honest "when to use which" note.

**Important project rule:** Work inside your existing `RetailMind` repository. Reuse the same target definition, train/test split, preprocessing decisions, and evaluation metric from your earlier churn work wherever possible. Do not fit preprocessing on the test set.

## 0. Environment setup

Use only today's project libraries: **TensorFlow / Keras**, plus the Python data-handling and evaluation libraries already used in your project (pandas, NumPy, scikit-learn, matplotlib).

If TensorFlow is not installed, run the install command in a terminal or a notebook cell, then restart the kernel.

In [ ]:
# If TensorFlow is missing, uncomment and run this line:
# %pip install tensorflow

import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)

## 1. Warm-up: MNIST handwritten digit demo

MNIST contains 28 × 28 grayscale images of handwritten digits (0–9). This small demo helps you understand:
- input image shape,
- normalization,
- Dense layers,
- softmax output for 10 classes,
- compiling, training and evaluating a Keras model.

This is a demonstration only. The RetailMind business model below uses the Telco churn dataset.

In [ ]:
# Load MNIST directly through Keras
(x_train_mnist, y_train_mnist), (x_test_mnist, y_test_mnist) = (
    keras.datasets.mnist.load_data()
)

print("Training images:", x_train_mnist.shape)
print("Training labels:", y_train_mnist.shape)
print("Test images:", x_test_mnist.shape)

In [ ]:
# View a few training images
plt.figure(figsize=(8, 3))

for i in range(5):
    plt.subplot(1, 5, i + 1)
    plt.imshow(x_train_mnist[i], cmap="gray")
    plt.title(f"Label: {y_train_mnist[i]}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Pixel values are 0 to 255. Neural networks usually train better
# when inputs are scaled to a small range such as 0 to 1.
x_train_mnist = x_train_mnist.astype("float32") / 255.0
x_test_mnist = x_test_mnist.astype("float32") / 255.0

print("Minimum pixel:", x_train_mnist.min())
print("Maximum pixel:", x_train_mnist.max())

In [ ]:
# A simple ANN: Flatten image -> Dense -> Dense output
mnist_model = keras.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax")
])

mnist_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

mnist_model.summary()

In [ ]:
mnist_history = mnist_model.fit(
    x_train_mnist,
    y_train_mnist,
    epochs=5,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

mnist_test_loss, mnist_test_accuracy = mnist_model.evaluate(
    x_test_mnist, y_test_mnist, verbose=0
)

print("MNIST test loss:", round(mnist_test_loss, 4))
print("MNIST test accuracy:", round(mnist_test_accuracy, 4))

### Check your understanding

- `Flatten()` changes a 28 × 28 image into a vector of 784 values.
- `Dense(128, activation="relu")` is a hidden layer with 128 neurons.
- `Dense(10, activation="softmax")` returns probabilities for 10 digit classes.
- `sparse_categorical_crossentropy` is suitable because labels are integer class IDs (0–9).

## 2. Load the Telco Customer Churn dataset

Dataset specified in the Day 5 brief: `blastchar/telco-customer-churn` on Kaggle.

Download `WA_Fn-UseC_-Telco-Customer-Churn.csv` from Kaggle and place it in your project's data folder. Update `DATA_PATH` if your folder/file name differs.

If you already have a cleaned/prepared Day 3/4 dataset, use the same source data and target definition so the comparison remains fair.

In [ ]:
DATA_PATH = "data/WA_Fn-UseC_-Telco-Customer-Churn.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found at: {DATA_PATH}\n"
        "Download the Telco Customer Churn CSV and update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\\nTarget distribution:")
print(df["Churn"].value_counts(dropna=False))

## 3. Basic data cleaning and target preparation

The target is `Churn`:
- `No` → 0
- `Yes` → 1

`customerID` is an identifier, not a useful predictive feature, so we drop it. `TotalCharges` is often stored as text; convert it to numeric. Invalid or blank values become missing and will be imputed using the training data only.

In [ ]:
data = df.copy()

# Remove customer identifier
if "customerID" in data.columns:
    data = data.drop(columns=["customerID"])

# Convert TotalCharges to numeric (blank strings become NaN)
if "TotalCharges" in data.columns:
    data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")

# Convert target labels to 0 and 1
data["Churn"] = data["Churn"].map({"No": 0, "Yes": 1})

# Remove rows with missing target, if any
data = data.dropna(subset=["Churn"])
data["Churn"] = data["Churn"].astype("int32")

X = data.drop(columns=["Churn"])
y = data["Churn"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("Churn rate:", round(y.mean(), 4))

## 4. Train/test split and preprocessing

We use a stratified split so the churn proportion is similar in train and test sets.

- Numeric columns: median imputation + standard scaling.
- Categorical columns: most-frequent imputation + one-hot encoding.

**Leakage prevention:** the preprocessing pipeline is fitted only on `X_train`, never on `X_test`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

X_train_processed = np.asarray(X_train_processed, dtype="float32")
X_test_processed = np.asarray(X_test_processed, dtype="float32")
y_train_array = np.asarray(y_train, dtype="float32")
y_test_array = np.asarray(y_test, dtype="int32")

print("Processed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

> Compatibility note: if your scikit-learn version does not recognize `sparse_output=False`, replace it with `sparse=False` in `OneHotEncoder`.

## 5. Build the ANN for churn prediction

This is a binary classification problem, so the output layer has:
- one neuron,
- sigmoid activation,
- predicted probability between 0 and 1.

We use ReLU in hidden layers and Adam as the optimizer.

In [ ]:
input_features = X_train_processed.shape[1]

ann_model = keras.Sequential([
    layers.Input(shape=(input_features,)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

ann_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall")
    ]
)

ann_model.summary()

## 6. Train with EarlyStopping

An **epoch** is one complete pass through the training data. Early stopping monitors validation loss and stops training if it does not improve for several epochs.

We keep a validation split from the training data. The test set remains untouched until final evaluation.

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = ann_model.fit(
    X_train_processed,
    y_train_array,
    validation_split=0.20,
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

print("Training stopped after", len(history.history["loss"]), "epochs")

In [ ]:
# Plot training and validation loss
plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Binary crossentropy loss")
plt.title("ANN training vs validation loss")
plt.legend()
plt.tight_layout()
plt.show()

### How to interpret the loss plot

- Both losses decreasing: the model is learning.
- Training loss decreasing while validation loss rises: possible overfitting.
- Both losses staying high: possible underfitting or an optimisation/data issue.
- EarlyStopping restores the weights from the epoch with the best monitored validation loss.

## 7. Evaluate the ANN on the held-out test set

The model predicts probabilities. For a simple baseline, we convert probabilities to class labels using a threshold of 0.5.

Do not assume 0.5 is always the best business threshold. Churn intervention costs and the cost of missing a customer may justify choosing a different threshold using validation data.

In [ ]:
ann_probabilities = ann_model.predict(X_test_processed, verbose=0).ravel()
ann_predictions = (ann_probabilities >= 0.5).astype("int32")

ann_results = {
    "Accuracy": accuracy_score(y_test_array, ann_predictions),
    "Precision": precision_score(y_test_array, ann_predictions, zero_division=0),
    "Recall": recall_score(y_test_array, ann_predictions, zero_division=0),
    "F1": f1_score(y_test_array, ann_predictions, zero_division=0),
    "ROC_AUC": roc_auc_score(y_test_array, ann_probabilities)
}

print(pd.Series(ann_results).round(4))
print("\\nClassification report:")
print(classification_report(y_test_array, ann_predictions, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test_array, ann_predictions)

plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title("ANN Confusion Matrix")
plt.xlabel("Predicted label")
plt.ylabel("Actual label")
plt.xticks([0, 1], ["No churn", "Churn"])
plt.yticks([0, 1], ["No churn", "Churn"])

for (i, j), value in np.ndenumerate(cm):
    plt.text(j, i, str(value), ha="center", va="center")

plt.tight_layout()
plt.show()

## 8. Compare ANN with your Phase 1 / Day 3–4 champion

**Important:** A valid head-to-head comparison requires both models to use the same:
- target definition,
- held-out test rows,
- train/test split,
- evaluation metric,
- and (where applicable) threshold-selection policy.

The following cell creates a comparison using your actual previous champion's test predictions. Update the variable names to match your earlier notebook. Do not enter guessed or invented metrics.

In [ ]:
# OPTION A: If you saved the previous champion's test-set predictions:
#
# champion_probabilities = your_saved_champion.predict_proba(X_test)[:, 1]
# champion_predictions = (champion_probabilities >= 0.5).astype(int)
#
# IMPORTANT:
# X_test here must represent exactly the same test customers used by
# the ANN comparison. If your earlier project used a different split,
# recreate a common split and evaluate both models on it.

# OPTION B: If you have already recorded the champion metrics,
# enter the real values from your earlier evaluation below.
# Leave as None until you have verified them.

champion_results = None

# Example structure only (replace with your actual measured metrics):
# champion_results = {
#     "Accuracy": 0.80,
#     "Precision": 0.60,
#     "Recall": 0.55,
#     "F1": 0.57,
#     "ROC_AUC": 0.82
# }

if champion_results is None:
    print("ANN results:")
    display(pd.Series(ann_results, name="ANN").round(4))
    print("Add your verified Phase 1 champion metrics to champion_results for comparison.")
else:
    comparison = pd.DataFrame({
        "Classical ML champion": champion_results,
        "ANN": ann_results
    })
    display(comparison.round(4))

### Record your conclusion

Answer these in your project notes:
1. Did ANN improve ROC-AUC over the classical champion?
2. Did ANN improve recall for churn customers?
3. Did the training/validation loss suggest overfitting?
4. Was the improvement large enough to justify additional complexity?
5. Which model would you deploy, and why?

Do not claim ANN is better just because it is deep learning. Choose based on measured test performance, reliability, complexity and business needs.

## 9. Save the model and preprocessing objects

The Keras model is saved in `.keras` format. The fitted preprocessing pipeline is also required at inference time because new customer rows must be transformed in exactly the same way as training data.

This notebook saves the preprocessor using Python's `pickle`. Only load pickle files that you created and trust.

In [ ]:
import pickle

os.makedirs("models", exist_ok=True)

ann_model.save("models/retailmind_churn_ann.keras")

with open("models/retailmind_churn_preprocessor.pkl", "wb") as file:
    pickle.dump(preprocessor, file)

print("Saved ANN model and fitted preprocessor.")

In [ ]:
# Test that the saved model can be loaded
loaded_ann_model = keras.models.load_model("models/retailmind_churn_ann.keras")

sample_probabilities = loaded_ann_model.predict(X_test_processed[:5], verbose=0).ravel()
print("First 5 churn probabilities:", np.round(sample_probabilities, 4))

## 10. Example prediction for new customer data

Use the original input columns, not already one-hot-encoded columns. The fitted preprocessor transforms the new row before the ANN predicts.

In [ ]:
# Take one row from the held-out test features as a demonstration.
# This is an existing customer row, not a new real-world customer.
sample_customer = X_test.iloc[[0]]

sample_processed = preprocessor.transform(sample_customer)
sample_processed = np.asarray(sample_processed, dtype="float32")

churn_probability = float(
    loaded_ann_model.predict(sample_processed, verbose=0)[0][0]
)

predicted_class = 1 if churn_probability >= 0.5 else 0

print("Churn probability:", round(churn_probability, 4))
print("Predicted churn:", "Yes" if predicted_class == 1 else "No")

## 11. README update — Day 5

Add a section like this to your RetailMind README. Fill in the actual values from your notebook; do not leave placeholder claims in the final README.

### ANN churn model
- Architecture: Dense(64, ReLU) → Dense(32, ReLU) → Dense(1, Sigmoid)
- Preprocessing: median imputation + standard scaling for numeric features; most-frequent imputation + one-hot encoding for categorical features.
- Training: Adam optimizer, binary crossentropy, EarlyStopping on validation loss.
- Evaluation: accuracy, precision, recall, F1 and ROC-AUC on the held-out test set.
- Model artifact: `models/retailmind_churn_ann.keras`
- Preprocessor artifact: `models/retailmind_churn_preprocessor.pkl`

### When to use which
Use the ANN only if the measured improvement over the classical champion justifies the added complexity and operational cost. For structured tabular data, classical tree-based models can remain strong, faster and easier to explain. Record the actual comparison and your reason for selecting the final model.

## 12. Day 5 completion checklist

- [ ] TensorFlow imports successfully.
- [ ] MNIST demo runs and test accuracy is recorded.
- [ ] Telco CSV loads successfully.
- [ ] Target and features are prepared correctly.
- [ ] Preprocessing is fitted only on training data.
- [ ] ANN architecture is created and summarized.
- [ ] Model trains with EarlyStopping.
- [ ] Training/validation loss plot is inspected.
- [ ] Test metrics and confusion matrix are recorded.
- [ ] ANN is compared fairly with the Phase 1 champion.
- [ ] Model and preprocessor are saved.
- [ ] README includes honest results and a "when to use which" note.
- [ ] Changes are committed and pushed to Git.

### Suggested Git commands

Run these from the root of your existing RetailMind repository:

```bash
git status
git add .
git commit -m "Day 5: add Keras ANN churn model"
git push
```

**End of Day 5.** Keep your notebook, model, preprocessor and README changes in the same RetailMind repository.